# LSN-004｜状态与时钟
**对应：RMD-003A · PFR1/PDP1 · 为 FR2/DP2 做准备**

## 本课问题
软件里的变量可以“记住”上一时刻的值。数字电路怎样记住上一时刻的膜电位？

> 本课唯一主要新概念：**digital state + clock。**

这一课先建立直觉，不要求会 SystemVerilog。

## 先看没有记忆的计算
组合逻辑只关心“现在的输入是什么”。输入一变，输出也跟着变；它本身不保存过去。

In [ ]:
def combinational_adder(a, b):
    return a + b

for a, b in [(1, 2), (4, 5), (10, -3)]:
    print(a, '+', b, '=', combinational_adder(a, b))

## 再看有状态的系统
下面用 Python 模拟一个 clocked register。重点不是 Python 语法，而是规则：**只有 tick 到来时，state 才更新。**

In [ ]:
state = 0
next_values = [3, 7, 2, 9]

for cycle, next_value in enumerate(next_values):
    before = state
    state = next_value  # imagine: update only on the clock edge
    print(f'cycle={cycle}: before={before}, after_clock={state}')

## 把它变成一个很像神经元的 accumulator
register 保存过去；加法器计算 next state；comparator 判断 threshold。

In [ ]:
state = 0
threshold = 4
inputs = [1, 1, 1, 1, 2, 2]

for cycle, x in enumerate(inputs):
    before = state
    next_state = state + x
    spike = next_state >= threshold
    state = 0 if spike else next_state
    print(f'cycle={cycle}: state={before}, input={x}, next={next_state}, spike={spike}, stored={state}')

## 软件 → 硬件的第一张翻译表

| 软件直觉 | 硬件直觉 |
|---|---|
| variable | register / RAM state |
| `if` | comparator + mux/control |
| function | module + interface |
| loop | parallel hardware 或 time multiplexing |
| 一次迭代 | 不一定等于一个 clock cycle |

最重要的一句：**RTL 不是“按顺序运行的软件”，而是在描述每个时钟边沿之间电路如何计算、边沿到来时哪些状态被保存。**

## AI Task
让 AI 用不超过 10 句话解释 combinational logic、register、clock edge，并要求它把上面的 accumulator 画成 `register → adder → comparator → mux → register` 数据流。暂时不要让它生成完整 LIF RTL。

## Human Check
- 哪个值是真正的 state？
- `next_state` 和 `state` 为什么不是同一个概念？
- 如果 input 在两个 clock edge 之间变化，什么东西应该保持不变？
- 为什么 Python `for` loop 不能直接等同于硬件里的“一个周期做一次”？

## Engineering Handoff
下一步 RMD-003A 的三个微实验将把这些直觉变成真实数字硬件：combinational adder、clocked counter、accumulator + threshold。完成这些以后才进入正式 LIF RTL。

## Exit Ticket
你能看一个简单时序表指出 state 在何时变化，并解释 combinational 与 sequential 的区别。